# Data Description
- Background: 第九輪10人的具名政府人物／職務名錄核對；9人跨公職生日、性別及申報學歷，1人政府調解委員明載性別／簡歷。
- date received: 新HTTP、工具摘錄及ZIP行政區碼抽出20261019；既有來源原保存日期見先前manifest，不宣稱本輪新下載。
- Path to data file: PINS精確檔案及hash；原網址見來源manifest與處理設定。CEC來源 https://db.cec.gov.tw/ 。
- Unit of observation: 原始HTTP回應、人物頁／名錄、CEC候選人列、行政區碼列、來源metadata分開。
- Sample period: 政府網站2026-10-19保存或本輪重閱，名錄頁可能最後更新2023–2025；歷史申報為2022，不能視為2026戶籍或學位重認證。
- Known issues: 官方名錄可能只有具名職務，不是完整履歷。TPP直接HTTP403另保存，工具可見段落是手動存下的摘錄、不是原HTML；Blogspot工具開啟失敗，但普通HTTP實際200內文成功，分方法記錄。政府頁及CEC都需定位個人區塊，不把同頁其他人資料套入。黃宣寓學歷與生日未知，調解委員不是民選職務。
- Definition for each variable: CEC無表頭0–4地域／選區碼、5號次、6姓名、7政黨碼、8性別碼、9民國生日、10當年年齡、11出生地、12申報學歷級別、13現任旗標、14當選符號、15其他旗標。elbase為5個地域碼加地名。manifest的url/status/path/sha256/retrieval_method為實際擷取與身分；excerpt.visible_text只保存工具實際可見片段。人工審閱設定在data/code，非原始來源。
- Data path: PINS逐檔列示。
- Role: raw HTTP bytes / unchanged ZIP extraction / source metadata / labelled web excerpt。
- Verification status: 下方只讀存在、SHA與格式；不改來源。
- SHA-256: PINS逐檔列示。
- Next processing notebook: [municipal_roles_round9_20261019.ipynb](../code/municipal_roles_round9_20261019.ipynb)。


In [1]:
from pathlib import Path
import pandas as pd,json,hashlib,csv
from bs4 import BeautifulSoup
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
PINS={'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/cec_all_offices/101_elcand.csv': '35853ac9d8a7eee4e9882bc04148462876110dcdb3cf0cdc11af0908e1c800f0', 'data/raw/cec_all_offices/103_elcand.csv': '46773806f1b6ef6389e632be7c10ea59bcb5c3afff3ad2715787974eafe494f0', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv': '4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79', 'data/raw/municipal_roles_round9_20261019/82e3fb55c475911e453d.html': 'ee5eccb74a99bc10078257e62d2a98fc4986eab743c853b7ce5f9debb56ab3e3', 'data/raw/municipal_roles_round9_20261019/6556882ec6ac55681d90.html': '027ddae3c619ea334c71cf921f67e0a119822e547f351f40b894d9a37eb12a9e', 'data/raw/municipal_roles_round9_20261019/d56e6a8a8d5ccaf32087.html': 'e8aa0d5deac638ff4b1be04646df7c2795520e1b2d2c38fe44bca6b5e460be09', 'data/raw/municipal_roles_round9_20261019/2022_R1_elbase.csv': '96ee98216f80e97cee27c8edbbbab35c240717bdd2ecf5eef4edaea4b45d36ee', 'data/raw/municipal_roles_round9_20261019/2022_D1_elbase.csv': '780f22895ffc9db630f92733fce3defbe913e6c936ca64cecf8860e34df0768a', 'data/raw/municipal_roles_round9_20261019/manifest.json': 'b32a0ca59f85bfd184e2f81af88e5c416430cc927196b280f238d31b7c2fb820', 'data/raw/municipal_roles_round9_20261019/extraction_manifest.json': '7f036b1e2f81e98e11cd54e36505f2c67b12ffc29ac7683dba8885a58c113b5f', 'data/raw/municipal_roles_round9_20261019/tpp_zhuang_web_excerpt.json': '02bb213dd07af91d2090223b74aea4164f51d02196faab68f5a3e0fe976c2a4f', 'data/raw/career_expansion_20261019/b587a5e46c6e1f6751ab.source': '546bf7a06c22f241135b6d64454d4d79615684ec3cb5f7679f053b59361311de', 'data/raw/career_expansion_20261019/e3a22c5d3489daf8cda3.source': '6ef3258faceba1559d55f5b3b33bedb3b5c5913ff69785351ffb4e7130fd67fb', 'data/raw/career_expansion_20261019/ae8adaeaad8ae7afaa36.source': '4cf7c9437b56e9757ba167f8f37a7e8b9375beac6b35b0e4566509fe28c322da', 'data/raw/career_expansion_20261019/64937ab1e916ff2df53f.source': 'a82ac876aea775810ba88270c917de751a00500e0b5af08d504ff6fbcc85b67e', 'data/raw/career_expansion_20261019/90edbc3b20226a41f3a8.source': 'ccc5959c2ab818c43776497d2c8e631553a2203097dabd8b1932f06063a6b66e', 'data/raw/career_expansion_20261019/0912e18a119d299531bf.source': 'b9f023b3b2aa30d147b291e3eaaa717987ac5ad5a381bb3ddde304ebaba73784', 'data/raw/career_expansion_20261019/606002eb4a668a9a18ea.source': '6a0a025642a80743ce91058c250cb8e3ae1e724569209f26667bd409624c8108', 'data/raw/career_expansion_20261019/6b8915ef8fb902feabfa.source': '6b20ad535e95fba23c662a8db3c4de71ef6d5677e95b9f2b93bcb02df11f67b9', 'data/raw/career_expansion_20261019/8fb25c97b1da43ae8bd0.source': 'dbd04ba156909aa18bc3e3d8e597a16d8a02a6ec2b13adb66ad1d0695633c019'}
for p,h in PINS.items():assert sha(p)==h,p
print('Raw source and metadata identities checked:',len(PINS))

Raw source and metadata identities checked: 22


## Change for Your Own Data
精確來源與hash，不以替代檔繞過檢查。

## Summary

### Sample

In [2]:
TABLES={}
for p in PINS:
 if p.endswith('.csv'):
  with (ROOT/p).open(encoding='utf-8-sig') as f:d=pd.DataFrame(list(csv.reader(f)))
  TABLES[p]=d;print(p);display(d.head(10))
for p in ['data/raw/municipal_roles_round9_20261019/manifest.json','data/raw/municipal_roles_round9_20261019/extraction_manifest.json']:
 print(p);display(pd.DataFrame(json.loads((ROOT/p).read_text())).head(10))

data/raw/cec_all_offices/101_elcand.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,09,007,00,010,0000,1,陳其平,1,1,0571024,54,金馬地區,高中(職),N,,
1,09,007,00,010,0000,2,林志東,1,1,0550819,56,金馬地區,專科,N,*,
2,09,007,00,020,0000,1,王建華,1,1,0640228,47,金馬地區,大學,N,,
3,09,007,00,020,0000,2,吳金平,1,1,0541106,56,金馬地區,碩士,N,*,
4,09,007,00,030,0000,1,陳樂禮,1,1,0610518,50,金馬地區,高中(職),N,*,
5,09,007,00,040,0000,1,林德建,1,1,0600504,51,金馬地區,高中(職),Y,*,
6,09,007,00,040,0000,2,曹恆勇,16,1,0640308,47,金馬地區,高中(職),N,,
7,09,007,00,040,0000,3,劉家發,999,1,0430314,68,金馬地區,高中(職),N,,
8,09,020,00,010,0000,1,李誠智,1,1,0650617,46,金馬地區,碩士,Y,*,
9,09,020,00,010,0000,2,歐陽彥木,999,1,0500105,61,金馬地區,大學,N,,


data/raw/cec_all_offices/103_elcand.csv


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,09,007,01,010,0000,1,李忠堅,1,1,0590222,52,金馬地區,高中(職),Y,*,
1,09,007,01,010,0000,2,曹爾淼,1,1,0650925,46,金馬地區,專科,Y,*,
2,09,007,01,010,0000,3,陳錦泰,1,1,0591108,51,金馬地區,高中(職),Y,*,
3,09,007,01,010,0000,4,李哲宇,16,1,0740602,37,臺灣省,專科,N,,
4,09,007,01,010,0000,5,陳聖泓,999,2,0790523,32,新北市,大學,N,*,
5,09,007,01,010,0000,6,陳彥錡,1,1,0770614,34,金馬地區,大學,N,*,
6,09,007,01,010,0000,7,林紹馨,999,1,0620713,49,金馬地區,高中(職),Y,*,
7,09,007,01,010,0000,8,陳則汎,1,1,0800311,31,金馬地區,大學,N,*,
8,09,007,01,010,0000,9,楊水英,1,2,0500411,61,金馬地區,高中(職),Y,,
9,09,007,01,020,0000,1,王冠予,1,1,0610403,50,金馬地區,大學,N,*,


data/raw/cec_all_offices/112_elcand.csv


...

,0,1,2,3,4,5
0,00,000,00,000,0000,全國
1,09,007,00,000,0000,連江縣
2,09,007,00,010,0000,南竿鄉
3,09,007,00,010,0001,介壽村
4,09,007,00,010,0002,復興村
5,09,007,00,010,0003,福沃村
6,09,007,00,010,0004,清水村
7,09,007,00,010,0005,仁愛村
8,09,007,00,010,0006,津沙村
9,09,007,00,010,0007,馬祖村


data/raw/municipal_roles_round9_20261019/manifest.json


,candidate_name,url,status,final_url,path,sha256,retrieval_method,checked_date
0,莊國輝,https://www.tpp.org.tw/election2026/candidated...,403,https://www.tpp.org.tw/election2026/candidated...,data/raw/municipal_roles_round9_20261019/65568...,027ddae3c619ea334c71cf921f67e0a119822e547f351f...,ordinary_requests_HTTP,2026-10-19
1,宋萬富,https://penghudaily.blogspot.com/2026/08/blog-...,200,https://penghudaily.blogspot.com/2026/08/blog-...,data/raw/municipal_roles_round9_20261019/82e3f...,ee5eccb74a99bc10078257e62d2a98fc4986eab743c853...,ordinary_requests_HTTP,2026-10-19
2,蔡志堅,https://udn.com/news/story/7324/9246669,200,https://udn.com/news/story/7324/9246669,data/raw/municipal_roles_round9_20261019/d56e6...,e8aa0d5deac638ff4b1be04646df7c2795520e1b2d2c38...,ordinary_requests_HTTP,2026-10-19


data/raw/municipal_roles_round9_20261019/extraction_manifest.json


,source_url,archive_member,archive_sha256,path,sha256,method
0,https://db.cec.gov.tw/,votedata/votedata/voteData/2022-111年地方公職人員選舉/R...,84740535b1f4a9a8fec8ebfe8f7577889837b7639cd7f5...,data/raw/municipal_roles_round9_20261019/2022_...,96ee98216f80e97cee27c8edbbbab35c240717bdd2ecf5...,unchanged extraction from preserved ZIP; no ne...
1,https://db.cec.gov.tw/,votedata/votedata/voteData/2022-111年地方公職人員選舉/D...,84740535b1f4a9a8fec8ebfe8f7577889837b7639cd7f5...,data/raw/municipal_roles_round9_20261019/2022_...,780f22895ffc9db630f92733fce3defbe913e6c936ca64...,unchanged extraction from preserved ZIP; no ne...


### Summary Stats

In [3]:
for p,d in TABLES.items():
 print(p,'rows',len(d),'columns',len(d.columns),'duplicate rows',int(d.duplicated().sum()));display(pd.DataFrame({'type':d.dtypes.astype(str),'empty':d.eq('').sum()}))
http=pd.DataFrame(json.loads((ROOT/'data/raw/municipal_roles_round9_20261019/manifest.json').read_text()));display(http.status.value_counts(dropna=False));assert len(http)==3
info=[]
for p in PINS:
 if p.endswith(('.source','.html')):
  s=BeautifulSoup((ROOT/p).read_text(errors='replace'),'html.parser');info.append({'path':p,'bytes':(ROOT/p).stat().st_size,'title':s.title.get_text(' ',strip=True) if s.title else '', 'tables':len(s.find_all('table'))})
display(pd.DataFrame(info));print('HTML titles/name hits do not verify all background claims.')

data/raw/cec_all_offices/101_elcand.csv rows 489 columns 16 duplicate rows 0


,type,empty
0,str,0
1,str,0
2,str,0
3,str,0
4,str,0
5,str,0
6,str,0
7,str,0
8,str,0
9,str,0


data/raw/cec_all_offices/103_elcand.csv rows 3225 columns 16 duplicate rows 0


,type,empty
0,str,0
1,str,0
2,str,0
3,str,0
4,str,0
5,str,0
6,str,0
7,str,0
8,str,0
9,str,0


data/raw/cec_all_offices/112_elcand.csv rows 14021 columns 16 duplicate rows 0


...

data/raw/municipal_roles_round9_20261019/2022_D1_elbase.csv rows 3437 columns 6 duplicate rows 0


,type,empty
0,str,0
1,str,0
2,str,0
3,str,0
4,str,0
5,str,0


status
200    2
403    1
Name: count, dtype: int64

,path,bytes,title,tables
0,data/raw/municipal_roles_round9_20261019/82e3f...,938001,宋萬富宣布參選縣議員 誓為白沙持續盡一份心力\n -\n 澎湖日...,0
1,data/raw/municipal_roles_round9_20261019/65568...,5572,Just a moment...,0
2,data/raw/municipal_roles_round9_20261019/d56e6...,186641,未獲民眾黨議員提名 竹市樹下里長退黨參選 | 桃竹苗 | 地方 | 聯合新聞網,0
3,data/raw/career_expansion_20261019/b587a5e46c6...,93425,內政部全球資訊網-中文網-鄉鎮市長,0
4,data/raw/career_expansion_20261019/e3a22c5d348...,132563,花蓮縣玉里鎮公所- 代表服務通訊,1
5,data/raw/career_expansion_20261019/ae8adaeaad8...,646984,調解委員會-花蓮縣吉安鄉公所全球資訊網,0
6,data/raw/career_expansion_20261019/64937ab1e91...,30719,代表名冊-馬公市民代表會,2
7,data/raw/career_expansion_20261019/90edbc3b202...,93330,內政部全球資訊網-中文網-村里長,0
8,data/raw/career_expansion_20261019/0912e18a119...,170474,宜蘭縣蘇澳鎮公所-通訊錄及代表簡介,0
9,data/raw/career_expansion_20261019/606002eb4a6...,123132,三星鄉第22屆鄉民代表,1


HTML titles/name hits do not verify all background claims.


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。